In [1]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

# Import the neet 2025 paper
initial_docs = (TextLoader("speech.txt")).load()

c:\Users\Abhishek Jha\miniconda3\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# convert into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 50)
final_documents = splitter.split_documents(initial_docs)

In [4]:
ollama_embedder = OllamaEmbeddings(model = "mxbai-embed-large", dimensions = 2000)

faiss_db = FAISS.from_documents(final_documents, ollama_embedder)

# Querying

In [5]:
query = "Where are the heavy oak doors?"

In [6]:
query_result = faiss_db.similarity_search(query)

In [7]:
query_result

[Document(id='597df9fa-3e7f-433a-90c3-395b783319c4', metadata={'source': 'speech.txt'}, page_content='The heavy oak doors of the auditorium sealed shut, plunging the crowded hall into a hushed,'),
 Document(id='bf4d43e3-4333-4336-a6b2-8053cfadaf33', metadata={'source': 'speech.txt'}, page_content="haunting sentence fade into the vast expanse of the vaulted ceiling, he didn't immediately step"),
 Document(id='3a7b59b2-87d0-4014-9563-304f588e596e', metadata={'source': 'speech.txt'}, page_content='home with a sudden, thunderous crescendo that practically vibrated the floorboards beneath their'),
 Document(id='09d72b1e-7f09-4610-b80d-f8355b5904bc', metadata={'source': 'speech.txt'}, page_content='chests. As his cadence began to accelerate, his hands—previously gripping the edges of the lectern')]

In [9]:
for result in query_result:
    print(result.page_content)

The heavy oak doors of the auditorium sealed shut, plunging the crowded hall into a hushed,
haunting sentence fade into the vast expanse of the vaulted ceiling, he didn't immediately step
home with a sudden, thunderous crescendo that practically vibrated the floorboards beneath their
chests. As his cadence began to accelerate, his hands—previously gripping the edges of the lectern


# FAISS as a retriever

In [10]:
faiss = faiss_db.as_retriever()

query_result_retriever = faiss.invoke(query)

In [11]:
query_result_retriever

[Document(id='597df9fa-3e7f-433a-90c3-395b783319c4', metadata={'source': 'speech.txt'}, page_content='The heavy oak doors of the auditorium sealed shut, plunging the crowded hall into a hushed,'),
 Document(id='bf4d43e3-4333-4336-a6b2-8053cfadaf33', metadata={'source': 'speech.txt'}, page_content="haunting sentence fade into the vast expanse of the vaulted ceiling, he didn't immediately step"),
 Document(id='3a7b59b2-87d0-4014-9563-304f588e596e', metadata={'source': 'speech.txt'}, page_content='home with a sudden, thunderous crescendo that practically vibrated the floorboards beneath their'),
 Document(id='09d72b1e-7f09-4610-b80d-f8355b5904bc', metadata={'source': 'speech.txt'}, page_content='chests. As his cadence began to accelerate, his hands—previously gripping the edges of the lectern')]

# Faiss Similarity Search with Score

While a standard search just returns documents, `similarity_search_with_score` returns the top documents alongside a **numerical distance score**.

* **Why it matters:** The score acts as a confidence metric. It allows you to build threshold logic (e.g., if the best score is too poor, tell the LLM to ignore the documents rather than hallucinate).
* **How to read it:** By default, LangChain's FAISS uses **L2 (Euclidean) Distance**. Therefore, **lower is better**. A score of `0.0` indicates a perfect match, while higher scores indicate lower relevance.

In [13]:
query

'Where are the heavy oak doors?'

In [14]:
query_and_score_results = faiss_db.similarity_search_with_score(query)

In [15]:
query_and_score_results

[(Document(id='597df9fa-3e7f-433a-90c3-395b783319c4', metadata={'source': 'speech.txt'}, page_content='The heavy oak doors of the auditorium sealed shut, plunging the crowded hall into a hushed,'),
  np.float32(0.84559447)),
 (Document(id='bf4d43e3-4333-4336-a6b2-8053cfadaf33', metadata={'source': 'speech.txt'}, page_content="haunting sentence fade into the vast expanse of the vaulted ceiling, he didn't immediately step"),
  np.float32(1.038672)),
 (Document(id='3a7b59b2-87d0-4014-9563-304f588e596e', metadata={'source': 'speech.txt'}, page_content='home with a sudden, thunderous crescendo that practically vibrated the floorboards beneath their'),
  np.float32(1.0930746)),
 (Document(id='09d72b1e-7f09-4610-b80d-f8355b5904bc', metadata={'source': 'speech.txt'}, page_content='chests. As his cadence began to accelerate, his hands—previously gripping the edges of the lectern'),
  np.float32(1.0937661))]

In [16]:
for doc, score in query_and_score_results:
    print(doc.page_content + "score = " + str(score))

The heavy oak doors of the auditorium sealed shut, plunging the crowded hall into a hushed,score = 0.84559447
haunting sentence fade into the vast expanse of the vaulted ceiling, he didn't immediately stepscore = 1.038672
home with a sudden, thunderous crescendo that practically vibrated the floorboards beneath theirscore = 1.0930746
chests. As his cadence began to accelerate, his hands—previously gripping the edges of the lecternscore = 1.0937661
